# 08 随机森林 Random Forest

依赖安装说明：`pip install numpy matplotlib scikit-learn`

随机森林是很多棵决策树的集成。每棵树只看一部分样本和一部分特征，最后投票或平均，通常比单棵树更稳定。


## 0. 学习目标和阅读地图

随机森林的重点是“用很多不完全一样的树降低方差”。你需要掌握：

1. bagging 为什么能让模型更稳定。
2. bootstrap 抽样和随机特征选择分别带来什么随机性。
3. 随机森林和单棵树相比牺牲了什么、获得了什么。
4. 如何理解特征重要性。


## 1. 数学逻辑

随机森林使用 bagging：

1. 从训练集有放回抽样，得到很多 bootstrap 数据集。
2. 每个数据集训练一棵树。
3. 分类时投票，回归时平均。

分类预测可以写成：

$$\hat y = \text{mode}\{T_1(x), T_2(x), \cdots, T_B(x)\}$$

随机性降低了树之间的相关性，平均后方差下降。


## 1.1 推导拆开看：为什么平均能降方差

如果有很多个误差不完全相关的模型，对它们取平均可以降低波动。回归时可以写成：

$$\hat f(x)=\frac{1}{B}\sum_{b=1}^{B}T_b(x)$$

分类时是投票：

$$\hat y=mode(T_1(x),\cdots,T_B(x))$$

关键不是“树越多越神奇”，而是每棵树要有一定准确性，同时彼此差异足够大。bootstrap 和 `max_features` 就是在制造这种差异。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(42)
X, y = make_moons(n_samples=300, noise=0.3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


## 1.2 随机森林训练时发生了什么

每棵树看到的数据都略有不同：

- 样本层面：bootstrap 有放回抽样。
- 特征层面：每次切分只考虑部分特征。

所以森林里的树会形成不同的错误模式。最终投票可以抵消单棵树的偶然错误。


In [ ]:
# 从零感受 bagging：训练多棵浅树，然后投票
rng = np.random.default_rng(42)
trees = []
for b in range(25):
    ids = rng.integers(0, len(X_train), size=len(X_train))
    tree = DecisionTreeClassifier(max_depth=4, max_features=1, random_state=b)
    tree.fit(X_train[ids], y_train[ids])
    trees.append(tree)

all_preds = np.array([tree.predict(X_test) for tree in trees])
bagging_pred = []
for col in all_preds.T:
    bagging_pred.append(Counter(col).most_common(1)[0][0])
bagging_pred = np.array(bagging_pred)

print('单棵树 accuracy:', round(accuracy_score(y_test, trees[0].predict(X_test)), 3))
print('bagging accuracy:', round(accuracy_score(y_test, bagging_pred), 3))


## 1.3 从零实现代码怎么读

从零版本用 `DecisionTreeClassifier` 模拟 bagging：

1. `rng.integers` 生成 bootstrap 样本索引。
2. 每棵树只在自己的抽样数据上训练。
3. `all_preds.T` 收集每个测试样本被所有树如何预测。
4. `Counter(...).most_common(1)` 做多数投票。

这展示了随机森林的主体思想，真实 `RandomForestClassifier` 还会在每次切分时随机选择候选特征。


In [ ]:
model = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('RandomForest accuracy:', round(accuracy_score(y_test, pred), 3))
print('feature importance:', np.round(model.feature_importances_, 3))

xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 180),
                     np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 180))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = model.predict(grid).reshape(xx.shape)
plt.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
plt.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap='coolwarm', edgecolor='k', s=24)
plt.title('随机森林决策边界')
plt.show()


In [ ]:
# 诊断：树数量对测试准确率的影响
n_estimators_grid = [1, 5, 10, 30, 80, 150, 250]
test_acc = []
for n_est in n_estimators_grid:
    m_rf = RandomForestClassifier(n_estimators=n_est, max_depth=5, random_state=42)
    m_rf.fit(X_train, y_train)
    test_acc.append(accuracy_score(y_test, m_rf.predict(X_test)))

plt.plot(n_estimators_grid, test_acc, marker='o')
plt.title('树数量与随机森林测试准确率')
plt.xlabel('n_estimators')
plt.ylabel('accuracy')
plt.show()


## 2.1 如何诊断随机森林

随机森林通常对超参数不如 boosting 敏感，但仍要关注：

- `n_estimators` 太少时结果不稳定。
- `max_depth` 太大时单棵树过深，但森林平均后通常缓解。
- 特征重要性可能偏向连续变量或取值多的变量。


## 2. 常见误区

- 随机森林通常强于单棵树，但可解释性会下降。
- 树很多不一定过拟合更严重，但训练和预测会更慢。
- 特征重要性会偏向取值多、切分机会多的特征。

## 3. 小实验

- 改 `n_estimators`，观察稳定性。
- 改 `max_depth`，观察过拟合。
- 对比单棵树和随机森林的决策边界。


## 5. 复习清单

- 随机森林 = 多棵随机化决策树投票。
- 它主要降低方差，不一定降低偏差。
- 比单棵树更稳，但解释性下降。
- 树数量增加通常更稳定，但计算成本更高。
